# Iterators, Generators and List Comprehentions


## Collection Types

&#x1F913; _Recap: What is printed here?_

In [ ]:
for item in [1,2,3]:
  print(item)
for item in (1,2,3):
  print(item)
for item in {1,2,3}:
  print(item)
for key in {1:'one', 2:'two', 3:'three'}:
  print(key)
for ch in 'python':
  print(ch)

&#x1F913; _Recap: `for` loops and lists: how does this work?_

In [ ]:
somelist = [1,2,3]

for item in somelist:
  print(item)

# is equivalent to:
i = 0
while i < len(somelist):
  item = somelist[i]
  print(item)
  i += 1

## Iterators

We can iterate over arbitrary collections.

In [ ]:
somecollection = ["a", "b", "c"]
for item in somecollection:
  print(item)

### Going over some collection

Under the hood, Python uses an _iterator_ as follows (this is also an example of how Python uses exceptions not just for things that go wrong but also for message passing).

* In the example below, in line 1, an iterator over `somecollection` is created.
Argument of `iter` needs to be _iterable_.
* `next(it)` returns the next item. It raises a `StopIteration` if there are no further elements, which means we jump out of the `while` loop.
* Iterators allow to iterate over the elements of arbitrary collection types (lists, sets, ...). You can think of them as pointers to the next element.

In [ ]:
it = iter(somecollection) # creates an iterator
while True:
  try:
    item = next(it) # returns the next element or throws a StopIteration
  except StopIteration:
    break
  print(item)

<img src="https://github.com/annefried/python-course/blob/main/images/11/iterator-hiker.png?raw=true" alt="a hiker going from one item to the next in a list until they reach a stop sign" width="600px" />

&#x1F913; _Execute the following code to see how `next` works._

In [ ]:
it = iter([1, 2, 3])
first  = next(it)
second = next(it)
third = next(it)
print(first, second, third)

In [ ]:
# Try executing next again:
next(it)

### Implementation in Python

<div class="alert alert-block alert-info">

Iterators are classes that implement the `__iter__` and the `__next__` methods.
* `__next__(self)` is called by `next()` and returns the next element or raises `StopIteration`.
* `__iter__(self)` is called by `iter()` and usually returns the iterator itself.
* An object `obj` is _iterable_ if it supports `iter(obj)`, i.e., if `__iter__(self)` is implemented and returns an iterator.
</div>

Iterators are an example of _duck typing_. Recall: The idea is that it does not matter what the type of some data is. In fact, there is no actual class called `Iterator`. It just matters what one can do with the data, i.e., what kind of methods are implemented.

<img src="https://github.com/annefried/python-course/blob/main/images/09/duck.png?raw=true" alt="image of a duck" width="150px"/>

The important behavior of iterators is that they store their current position (kind of like a pointer) within a collection, and move to the next position when requested.

We can implement an iterator over a list ourselves like this:

In [ ]:
class ListIterator:
  def __init__(self, lis):
    self.lis = lis
    self.index = -1
  def __iter__(self): # needs to return itself by definition
    return self
  def __next__(self): # returns next element
    print("The next element has been requested")
    self.index += 1
    if self.index >= len(self.lis):
      print("Sorry, already at the end.")
      raise StopIteration
    return self.lis[self.index]

In [ ]:
mylist = [42, 43, 44]
myiterator = ListIterator(mylist)
for item in myiterator:
    print(item)

Here is another example. Recall that after exiting the `with` block, the file is automatically closed.

In [ ]:
class FileIterator:
  def __init__(self, file):
    self.file = file
  def __iter__(self):
    return self
  def __next__(self):
    line = self.file.readline()
    if line == '':
      raise StopIteration
    return line

with open('example.txt') as f:
  for line in FileIterator(f):
    print(line)
# file is automatically closed

### Exercises

Many different collection types can be turned into iterators with `iter()`. Thus, they are all iterable.
Iterators are clearly iterable (you can iterate over them), so they also need to implement `iter()`. However, for something that's already an iterator (i.e. it also has `next()`), we don't want to create a _new_ iterator object. So `iter()` should just return `self` in these cases.

In `for`-loops, Python implicitly calls `iter()` on the iterable that we loop over, no matter if it is already an iterator or an iterable collection. 
Note that we can create multiple iterators with the same collection. The collection does not get copied, it is always the same collection object that holds the data internally. But if we have multiple iterator objects, we can have multiple pointers to different positions at the same time. If we only have one iterator object, we only store one position.

This can lead to interesting effects.

What is the value of the result in the following two cases? First try to determine the result using pen and paper, then execute the code.

In [ ]:
# Version 1
result = 0
lis = [1,2,3]
for x in lis:
  for y in lis:
    result += x * y
print(result)

In [ ]:
# Version 2
result = 0
it = iter([1,2,3])
for x in it:
  for y in it:
    result += x * y
print(result)


In [ ]:
# What happens in this case? How can we fix it?
lis = [1,2,3]
for x in lis:
  lis.append(x)
print(lis)

### Limitations

- Iterators have no lenghts (`len(iterator)` does not work)
- Iterators cannot be copied (at least not easily)
- no slicing etc.

To turn an iterator into a new object, you can use the usual type casting functions:

In [ ]:
someiterator = iter([1,1,2,3,5,8])

# l = list(someiterator)
# print(l)
s = set(someiterator)
print(s)
t = tuple(someiterator)
print(t)

# Note that only one of these works at a time - once an iterator is "used up" (has raised StopIteration), 
# it outputs no more elements.

### Itertools
Python has an [itertools](https://docs.python.org/3/library/itertools.html) module that provides
functionality for creating fast, memory-efficient iterators. The three most important functions from itertools are accessible directly.
All of them return iterators.
* `range([start,], stop [, step])`
* `enumerate(iterable)`
* `zip(it1, it2, ...)`

Work with 1-2 partners, make sure you understand how the three most important functions from `itertools` work.
They are directly accessible in Python, i.e., you do not need to import the module.
Collect several examples per function and document its functionality in the Jupyter notebook.

__Recommended reading:__
[Python Like You Mean It: Python's "Itertools"](https://www.pythonlikeyoumeanit.com/Module2_EssentialsOfPython/Itertools.html)

In [ ]:
# range

In [ ]:
# enumerate

In [ ]:
# zip

### Further Exercises

(In case you have time in class or you want to just gain a deeper understanding / test your understanding).

Reimplement the builtin iterator `enumerate(iterable)`.

In [ ]:
class MyEnumerate:
    pass

for (i, ch) in MyEnumerate("Python"):
    print(i, ch)

Reimplement the following function so that it also works iterators.
```
def avg(seq):
  return sum(seq) / len(seq)

print(avg([1,2,3,4])) # prints 2.5
print(avg(iter([1,2,3,4]))) # error
```

A straightforward way to make avg work also with iterators would be:

```
    def avg(iterable):
        sequence = list(iterable)
        return sum(sequence) / len(sequence)
```

However, this implemenation would be quite inefficent if list(iterable) results
in a very long list.

Reimplement the avg function so that it can be applied to iterators, in a way
that uses iterators directly (without computing a list)

In [ ]:
def avg(someiterator):
    pass

print(avg(iter([1,2,3,4])))

### Summary: Iterators

* Can be used instead of lists/sets:
  * `for  x in iterator: ...`
  * `if x in iterator: ...`
* Some limitations:
  * they have no length (`len(iterator)`  does not work)
  * they cannot be copied (at least not easily)
  * no slicing etc.
* To turn an iterator into a new object:
```
l = list(someiterator)
s = set(someiterator)
t = tuple(someiterator)
```

## Generators

Iterables store all their "contents" in memory. A _generator_ is a special kind of function that leverages lazy evaluation: it generates and "returns" (_yields_) one item of a sequence at a time when `next()` is called on the generator object.

In [ ]:
def einmaleins(number):
  max_value = number * 10
  x = number
  while x <= max_value:
    yield x # "return" reference to value of x and wait
            # for next() to be called on generator object
    x += number
  # reached here: StopIteration

In [ ]:
dreimaldrei = einmaleins(3)
next(dreimaldrei)

In [ ]:
for number in dreimaldrei:
    print(number)

next(number)
# This raises a StopIteration: each generator works only once (like iterator objects)

__Exercise:__

Write a generator function that takes a list as an argument and returns pairs (index, value), i.e., that reimplements the `enumerate` function.

In [ ]:
def my_enumerate_generator(somelist):
    yield None # change this!

g = my_enumerate_generator(["a", "b", "c", "d"])

## List Comprehensions

_List comprehensions_ are a concise way to create lists in Python.
This expression evaluates to a list of items obtained by evaluating `<expression>`
	`for every \texttt{<variable> in <iterable>`:

```
y = [x + 1 for x in [1, 2, 3]]
print(y) # [2, 3, 4]`
```

<div class="alert alert-block alert-info">

List comprehensions syntax:

```
result = [<expression> for <variable> in <iterable>]

# similar to:
result = []
for <variable> in <iterable>:
  result.append(<expression>)
```

Difference: variables introduced in a list comprehension are introduced in their own namespace, i.e., they are local to the list comprehension.
</div>

You can also nest several `for` loops:

In [ ]:
# Several for loops
result = [x + y for x in [1, 2, 3] for y in [4, 5, 6]]

result = []
for x in [1, 2, 3]:
  for y in [4, 5, 6]:
   result.append(x + y)
# [5, 6, 7, 6, 7, 8, 7, 8, 9]

&#x1F600; _Run and rewrite the following list comprehensions with regular `for` loops._

In [ ]:
r1 = [x for x in [2,3] if x % 2 == 0]

In [ ]:
r2 = [x + y for x in [2,3] if x % 2 == 0 for y in [5,6]]

In [ ]:
r3 = [x + y for x in [2,3] for y in [5,6] if x % 2 == 0]

### Set and Dictionary Comprehensions

They work like list comprehensions, but the outcome is a set or dictionary. Can you spot the differences?

In [ ]:
# Set Comprehension
s1 = {x for x in [1, 2, 2, 3, 1, 5, 6]}
s2 = {x for x in [1, 2, 2, 3, 1, 5, 6] if x % 2 == 0}
print(s1)
print(s2)

# Dictionary Comprehension
d = {x : 2 ** x for x in [1, 2, 3]}

In [ ]:
# Common use case for dictionary comprehensions: small indices in machine learning
labels = ["red", "green", "yellow", "blue"]
label2idx = {l : i for i, l in enumerate(labels)}
idx2label = {i : l for i, l in enumerate(labels)}

print("Which is the third label?", idx2label[2])
print("What is the index of label green?", label2idx["green"])


### Generator Expressions

_Generator expressions_ are a variant of list comprehensions: `(<expr> for <var> in <iterable>)` - note the parentheses around the expression.
They do not create a (potentially large) list objects but create the values on-the-fly.
Special case: The parentheses can be omitted if a generator expression is used as a single argument in a function call (as for `sum` below).

In [ ]:
# Generators as arguments
sum((x * x for x in [1, 2, 3]))
sum(x * x for x in [1, 2, 3])